#### 调用Deepseek API

In [2]:
import os
import json
import requests
from dotenv import load_dotenv

# 加载.env 文件中的环境变量
load_dotenv()       # 在当前目录寻找 .env 文件

api_key = os.getenv("DEEPSEEK_API_KEY")

if not api_key:
    raise ValueError("请在 .env 文件中设置DEEPSEEK_API_KEY")

# 设置端点url
url = "https://api.deepseek.com/chat/completions"


##### 1.基础调用

In [17]:
# 4. 构建消息数组
msg = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "你好"}
]

# 5. 构建请求体并转换为JSON字符串
payload = {
    "messages": msg,               # 必填字段，消息数组，包含content和role属性，至少一条消息
    "model": "deepseek-v4-flash",  # 必填字段，模型名称
    "stream": False                # 非流式，方便一次拿到完整响应
}

# 6. 设置请求头
headers = {
    "Content-Type": "application/json",         # 请求内容类型为JSON
    "Accept":'application/json',                # 接收的响应内容类型为JSON     
    "Authorization": f"Bearer {api_key}"        # API_keys,用于身份验证
}

# 7. 发送 POST 请求
response = requests.request("POST",url, headers=headers, json=payload)  # 用 json= 自动序列化，更简洁

# # 打印响应类型和内容
# print(response.text)

# 8. 处理响应
if response.status_code == 200:
    result = response.json()
    print(result)
    # 提取回复内容
    content = result["choices"][0]["message"]["content"]
    print("AI 回复：", content)
else:
    print(f"请求失败，状态码：{response.status_code}")
    print("错误详情：", response.text)

{'id': '47342eb1-93e4-4196-86eb-cb44edd25c5a', 'object': 'chat.completion', 'created': 1787193019, 'model': 'deepseek-v4-flash', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '你好！😊 很高兴见到你。有什么我可以帮你的吗？无论是回答问题、提供建议，还是陪你聊天，我都乐意效劳！', 'reasoning_content': 'The user greeted me in Chinese with "你好". I should respond in a friendly and helpful manner in Chinese. Let me craft a warm and inviting response that opens the door for further conversation.'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 89, 'completion_tokens': 69, 'total_tokens': 158, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 89}, 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e'}
AI 回复： 你好！😊 很高兴见到你。有什么我可以帮你的吗？无论是回答问题、提供建议，还是陪你聊天，我都乐意效劳！


##### 2.流式请求

In [20]:
# 4. 构建消息数组
msg = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "宋朝延续多少年，输出不超过50字"}
]

# 5. 构建请求体并转换为JSON字符串
payload = {
    "messages": msg,               # 必填字段，消息数组，包含content和role属性，至少一条消息
    "model": "deepseek-v4-flash",  # 必填字段，模型名称
    "stream": True,                # 非流式，方便一次拿到完整响应
    "max_tokens":60,
}

# 6. 设置请求头
headers = {
    "Content-Type": "application/json",         # 请求内容类型为JSON
    "Accept":'application/json',                # 接收的响应内容类型为JSON     
    "Authorization": f"Bearer {api_key}"        # API_keys,用于身份验证
}

# 逐块处理响应
try:
    response = requests.request("POST",url,headers=headers,json=payload,stream=True)    # 发送POST请求并获取响应
    response.raise_for_status()             # 检查HTTP响应状态，如果不是200则抛出异常
    for chunk in response.iter_lines():     # 逐行读取流式响应
        if chunk:
            decode_chunk = chunk.decode('utf-8').strip()    # 将字节六解码为字符串并去除首尾空白
            if decode_chunk.startswith("data:"):            # 判断是否为数据行
                data_str =  decode_chunk[6:]                # 提取数据内容（去掉前缀"data："）
                if data_str =="[DONE]":         # 流结束标识
                    break                       # 终止循环，结束读取
                try:
                    json_data =json.loads(data_str)     # 解析JSON数据
                    if "choices" in json_data:
                        content = json_data["choices"][0].get("delta",{}).get("content","")
                        print(content,end="",flush=True)        # 逐字输出，刷新缓冲
                except json.JSONDecodeError:
                    # 解析失败时忽略，继续处理下一块数据
                    continue 
            
except requests.exceptions.RequestException as e:
    print(f"\n请求失败:{e}")         # 打印错误信息



NoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNone

#### 3.system message

##### 3.1 设定身份

In [24]:
# 4. 构建消息数组
msg = [
    # {"role":"system","content":"你是一位优秀的脱口秀演员，请用幽默的方式回答用户问题。"},
    {"role":"user","content":"如何在30岁之前财务自由？"}
]

# 5. 构建请求体并转换为JSON字符串
payload = {
    "messages": msg,               # 必填字段，消息数组，包含content和role属性，至少一条消息
    "model": "deepseek-v4-flash",  # 必填字段，模型名称
    "stream": False                # 非流式，方便一次拿到完整响应
}

# 6. 设置请求头
headers = {
    "Content-Type": "application/json",         # 请求内容类型为JSON
    "Accept":'application/json',                # 接收的响应内容类型为JSON     
    "Authorization": f"Bearer {api_key}"        # API_keys,用于身份验证
}

# 7. 发送 POST 请求
response = requests.request("POST",url, headers=headers, json=payload)  # 用 json= 自动序列化，更简洁

# # 打印响应类型和内容
# print(response.text)

# 8. 处理响应
if response.status_code == 200:
    result = response.json()
    print(result)
    # 提取回复内容
    content = result["choices"][0]["message"]["content"]
    print("AI 回复：", content)
else:
    print(f"请求失败，状态码：{response.status_code}")
    print("错误详情：", response.text)

{'id': '634f2f9b-a900-449e-877c-bb15954ef02e', 'object': 'chat.completion', 'created': 1787196524, 'model': 'deepseek-v4-flash', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '先说一句实话：**30岁前财务自由，靠的不是“理财技巧”，而是“极高的收入 + 极高的储蓄率 + 极低的物欲 + 一点点运气”。** 这不是不可能，但确实是小概率事件。如果你想认真逼近，可以按下面这套逻辑来操作。\n\n---\n\n## 1. 先定义你要的“财务自由”\n\n财务自由不是“有几个亿”，而是：\n\n> **被动收入 ≥ 日常开销**\n\n按国际通用“4%法则”估算：\n\n> **需要本金 = 年开销 × 25**\n\n举例：\n\n- 如果你每年花 10 万：需要 250 万\n- 如果你每年只花 6 万：需要 150 万\n- 如果你每年花 30 万：需要 750 万\n\n所以，**降低欲望，直接降低财务自由的门槛。**\n\n---\n\n## 2. 30岁前实现的唯一公式\n\n如果你 22 岁毕业，到 30 岁只有 8 年。\n\n假设你目标本金 250 万：\n\n- 8 年存 250 万，平均每年要存 **31.25 万**\n- 也就是说，你的税后年收入至少 40–50 万，并且每年几乎不能乱花\n\n结论：  \n**普通上班族靠死工资，很难。必须“开源”到极致。**\n\n---\n\n## 3. 具体怎么做？三个核心：赚钱、省钱、投资\n\n### ① 把收入拉高一个量级\n\n这是最重要的。\n\n- 选择高薪行业：金融、互联网、AI、销售、出海、医疗等\n- 做到不可替代：成为前 20% 的工程师/销售/运营/设计师\n- 学会谈薪：跳槽比内部涨薪快\n- 做有杠杆的事：写代码、写书、做课程、做IP、做跨境贸易\n- 发展副业：不是“送外卖”，而是“可复制的副业”，比如自媒体、独立开发、咨询、小规模电商\n\n30岁前财务自由的人，大多不是“省出来的”，而是“赚出来的”。\n\n---\n\n### ② 储蓄率要变态高\n\n

##### 3.2 控制输出格式

In [26]:
# 4. 构建消息数组
msg = [
    {"role":"system","content":"你是一个结构化数据生成器，所有回答必须使用JSON格式，字段包括title和summary。"},
    {"role":"user","content":"请总结一下苹果的定义。"}
]

# 5. 构建请求体并转换为JSON字符串
payload = {
    "messages": msg,               # 必填字段，消息数组，包含content和role属性，至少一条消息
    "model": "deepseek-v4-flash",  # 必填字段，模型名称
    "stream": False                # 非流式，方便一次拿到完整响应
}

# 6. 设置请求头
headers = {
    "Content-Type": "application/json",         # 请求内容类型为JSON
    "Accept":'application/json',                # 接收的响应内容类型为JSON     
    "Authorization": f"Bearer {api_key}"        # API_keys,用于身份验证
}

# 7. 发送 POST 请求
response = requests.request("POST",url, headers=headers, json=payload)  # 用 json= 自动序列化，更简洁

# # 打印响应类型和内容
# print(response.text)

# 8. 处理响应
if response.status_code == 200:
    result = response.json()
    print(result)
    # 提取回复内容
    content = result["choices"][0]["message"]["content"]
    print("AI 回复：", content)
else:
    print(f"请求失败，状态码：{response.status_code}")
    print("错误详情：", response.text)

{'id': 'd80e237d-c8b5-4f86-9f34-dcb64a60e5a1', 'object': 'chat.completion', 'created': 1787207767, 'model': 'deepseek-v4-flash', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '{\n  "title": "苹果的定义",\n  "summary": "苹果是蔷薇科苹果属植物的果实，通常呈圆形，果皮颜色多为红、绿或黄色，果肉脆嫩多汁，味甜或略酸。它富含维生素、膳食纤维和抗氧化物质，既可生食，也可加工成果汁、果酱等。苹果广泛种植于温带地区，是全球消费量最大的水果之一。"\n}', 'reasoning_content': '我们根据要求，用JSON格式回答。title和summary。总结苹果的定义。'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 109, 'completion_tokens': 108, 'total_tokens': 217, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 109}, 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e'}
AI 回复： {
  "title": "苹果的定义",
  "summary": "苹果是蔷薇科苹果属植物的果实，通常呈圆形，果皮颜色多为红、绿或黄色，果肉脆嫩多汁，味甜或略酸。它富含维生素、膳食纤维和抗氧化物质，既可生食，也可加工成果汁、果酱等。苹果广泛种植于温带地区，是全球消费量最大的水果之一。"
}


##### 3.3 限制行为

In [28]:
# 4. 构建消息数组
msg = [
    {"role":"system","content":"你是一位严谨的AI助手，遇到你不知道的问题要明确说不知道，不能编造答案。"},
    {"role":"user","content":"请告诉我2027年火星移民的最新数据。"}
]

# 5. 构建请求体并转换为JSON字符串
payload = {
    "messages": msg,               # 必填字段，消息数组，包含content和role属性，至少一条消息
    "model": "deepseek-v4-flash",  # 必填字段，模型名称
    "stream": False                # 非流式，方便一次拿到完整响应
}

# 6. 设置请求头
headers = {
    "Content-Type": "application/json",         # 请求内容类型为JSON
    "Accept":'application/json',                # 接收的响应内容类型为JSON     
    "Authorization": f"Bearer {api_key}"        # API_keys,用于身份验证
}

# 7. 发送 POST 请求
response = requests.request("POST",url, headers=headers, json=payload)  # 用 json= 自动序列化，更简洁

# # 打印响应类型和内容
# print(response.text)

# 8. 处理响应
if response.status_code == 200:
    result = response.json()
    print(result)
    # 提取回复内容
    content = result["choices"][0]["message"]["content"]
    print("AI 回复：", content)
else:
    print(f"请求失败，状态码：{response.status_code}")
    print("错误详情：", response.text)

{'id': 'a3d2bb1e-f72c-4b85-8aaa-6627ee4374d5', 'object': 'chat.completion', 'created': 1787207927, 'model': 'deepseek-v4-flash', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '首先，我需要明确指出：**2027年属于未来，目前（2024年）尚未有任何关于“2027年火星移民”的实际数据。** \n\n作为严谨的AI，我不能编造未来的统计数据。截止到目前，人类的太空探索事实是：\n\n1.  **尚未有真人登陆火星**：人类至今为止最远的载人任务仍然停留在月球（1969-1972年）。\n2.  **火星上没有任何永久性人类定居点**：目前火星表面只有机器人探测器（如“毅力号”、“好奇号”、“祝融号”等），没有任何人类移民。\n\n关于您提到的**2027年**，目前的航天计划中并没有“移民火星”的既定官方计划。我们只能看到一些**规划中的里程碑**：\n\n-   **NASA（美国宇航局）**：计划在2027年执行“阿尔忒弥斯三号”任务，目标是将宇航员送上**月球南极**（注意，这是月球，不是火星）。\n-   **SpaceX（太空探索技术公司）**：马斯克设想的“星舰”飞船正在研发中，其长期目标是火星。但目前最乐观的预测是，最早可能在**2028年或2030年代**执行不载人的火星任务，载人火星任务更是要到2030年代中后期。\n-   **中国航天**：根据发布的规划，中国预计在**2030年左右**实现载人登月，而载人火星探测计划预计在**2033年左右**。\n\n**总结来说：** 如果你问的是“2027年移民火星的人口数量”，答案是**0**（且该目标在2027年不可能实现）。如果你对未来火星探索感兴趣，目前真正的技术转折点在于2027年的“星舰”轨道试飞以及“阿尔忒弥斯”登月计划。请以官方机构的实时发布为准，不要相信任何关于“2027年火星移民”的营销或夸大宣传。', 'reasoning_content': '1.  **分析用户的请求**：\n    *   主题：火星移民（火星移民）。\n    *   时间范围：2027年

##### 3.4 明确任务类型+深度思考标签

In [9]:
# 4. 构建消息数组
msg = [
    {"role":"system","content":"你是一个专业英语翻译助手，只负责将用户的内容翻译成英文，不要添加任何其他内容。"},
    {"role":"user","content":"深度学习模型的模型的训练需要大量标注数据。"}
]

# 5. 构建请求体并转换为JSON字符串
payload = {
    "messages": msg,               # 必填字段，消息数组，包含content和role属性，至少一条消息
    "model": "deepseek-v4-flash",  # 必填字段，模型名称
    "stream": False,                # 非流式，方便一次拿到完整响应
    "thinking": {"type": "enabled"},
    "reasoning_effort": "high"
}

# 6. 设置请求头
headers = {
    "Content-Type": "application/json",         # 请求内容类型为JSON
    "Accept":'application/json',                # 接收的响应内容类型为JSON     
    "Authorization": f"Bearer {api_key}"       # API_keys,用于身份验证

}

# 7. 发送 POST 请求
response = requests.request("POST",url, headers=headers, json=payload)  # 用 json= 自动序列化，更简洁

# # 打印响应类型和内容
# print(response.text)

# 8. 处理响应
if response.status_code == 200:
    result = response.json()
    print(result)
    # 提取回复内容
    content = result["choices"][0]["message"]["content"]
    thinking = result["choices"][0]["message"]["reasoning_content"]
    print("AI 思考：", thinking)
    print("AI 回复：", content)
else:
    print(f"请求失败，状态码：{response.status_code}")
    print("错误详情：", response.text)

{'id': 'aaa77ad9-edcf-4c0b-8ff2-81cf6cdef1aa', 'object': 'chat.completion', 'created': 1787214677, 'model': 'deepseek-v4-flash', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'The training of deep learning models requires a large amount of labeled data.', 'reasoning_content': '我们根据指令，将用户的中文内容翻译成英文。用户内容为“深度学习模型的模型的训练需要大量标注数据。”注意原文中“模型的模型”可能是重复，但按原意翻译。翻译成英文即可。'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 114, 'completion_tokens': 62, 'total_tokens': 176, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 47}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 114}, 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e'}
AI 思考： 我们根据指令，将用户的中文内容翻译成英文。用户内容为“深度学习模型的模型的训练需要大量标注数据。”注意原文中“模型的模型”可能是重复，但按原意翻译。翻译成英文即可。
AI 回复： The training of deep learning models requires a large amount of labeled data.
